In [4]:
import cv2
import os
import numpy as np
import pandas as pd

gt_dir = "Ground_Truth_Masks/" 
pred_mask_dir = "segmented_frames/" 

if not os.path.exists(gt_dir) or not os.path.exists(pred_mask_dir):
    print("❌ ERROR: Ensure 'Ground_Truth_Masks' and 'segmented_frames' folders exist.")
else:
    results = []
    thickness_kernel = np.ones((15, 15), np.uint8)

    gt_files = [f for f in os.listdir(gt_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    
    print(f"Found {len(gt_files)} Ground Truth frames for evaluation...")

    for fname in gt_files:
        pred_path = os.path.join(pred_mask_dir, fname)
        if not os.path.exists(pred_path):
            continue
            
        gt_img = cv2.imread(os.path.join(gt_dir, fname), 0)
        pred_img = cv2.imread(pred_path, 0)
        
        if gt_img is None or pred_img is None: continue
        
        h, w = gt_img.shape
        
        roi_mask = np.zeros((h, w), dtype=np.uint8)
        roi = np.array([[(int(w*0), int(h*1)), 
                         (int(w*0), int(h*0.5)),
                         (int(w*0.1), int(h*0.30)),
                         (int(w*0.9), int(h*0.30)), 
                         (int(w*1), int(h*0.5)),
                         (int(w*1), int(h*1))]], np.int32)
        cv2.fillPoly(roi_mask, roi, 1)

        gt_bin = (gt_img > 0).astype(np.uint8) * roi_mask
        pred_bin = cv2.dilate((pred_img > 0).astype(np.uint8), thickness_kernel, iterations=1)
        pred_bin = pred_bin * roi_mask

        TP = np.sum((pred_bin == 1) & (gt_bin == 1))
        TN = np.sum((pred_bin == 0) & (gt_bin == 0))
        FP = np.sum((pred_bin == 1) & (gt_bin == 0))
        FN = np.sum((pred_bin == 0) & (gt_bin == 1))
        
        acc = (TP+TN) / (TP+TN+FP+FN + 1e-6)
        iou = TP / (TP+FP+FN + 1e-6)
        prec = TP / (TP+FP + 1e-6)
        rec = TP / (TP+FN + 1e-6)
        f1 = 2 * (prec*rec) / (prec+rec + 1e-6)
        
        results.append([fname, acc, prec, rec, f1, iou])

    if not results:
        print("❌ No matching frames found between GT and Pred folders.")
    else:
        df = pd.DataFrame(results, columns=["Frame", "Accuracy", "Precision", "Recall", "F1", "IoU"])

        summary = df.mean(numeric_only=True).round(4)

        print("\n" + "="*50)
        print(" PERFORMANCE SUMMARY (AVERAGE) ")
        print("="*50)
        print(summary)

        print("\nDetailed Frame-wise results (First 5):")
        print(df.head().to_string(index=False))

Found 10 Ground Truth frames for evaluation...

 PERFORMANCE SUMMARY (AVERAGE) 
Accuracy     0.6620
Precision    0.5944
Recall       0.2857
F1           0.3583
IoU          0.2224
dtype: float64

Detailed Frame-wise results (First 5):
          Frame  Accuracy  Precision   Recall       F1      IoU
frame_00000.jpg  0.848070   0.065061 0.303241 0.107135 0.056599
frame_00050.jpg  0.625258   0.935692 0.288659 0.441206 0.283044
frame_00095.jpg  0.581616   0.755817 0.246471 0.371723 0.228293
frame_00110.jpg  0.654614   0.737897 0.293728 0.420193 0.265978
frame_00175.jpg  0.586863   0.518650 0.225398 0.314234 0.186405
